# 00 · Setup e Ingestão

Cria o bucket, cria os três datasets do Medallion, baixa o dataset do Kaggle e
sobe os arquivos para o Cloud Storage. Rode uma vez por projeto.

---

### A regra do holdout

O `fraudTrain.csv` vai para `raw/train/` e entra no BigQuery. O `fraudTest.csv`
vai para `holdout/` e não é carregado: ele cobre o período posterior a
21/06/2020, e será usado apenas no Trabalho 2. Se entrar na Bronze agora, deixa
de ser um conjunto não visto.

In [ ]:
import io
import os
import zipfile
from getpass import getpass
from pathlib import Path

import pandas as pd
from google.cloud import bigquery
from google.cloud import storage

In [ ]:
!pip install --quiet --upgrade google-cloud-bigquery google-cloud-storage kaggle db-dtypes

In [ ]:
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticado no Colab — use a conta dona do projeto")
except ImportError:
    print("Fora do Colab: usando as credenciais do ambiente")

## Configuração

Uma região para todos os serviços. Carga entre regiões diferentes falha, e o
streaming do Trabalho 2 depende da mesma colocação.

In [ ]:
PROJECT_ID  = "fraudflow-pdm-gps"
BUCKET_NAME = "fraudflow-pdm-gps-data"
REGION      = "us-central1"
BQ_LOCATION = "us-central1"

DATASETS = ["bronze", "silver", "gold"]

ARQUIVO_TRAIN   = "raw/train/fraudTrain.csv"
ARQUIVO_HOLDOUT = "holdout/fraudTest.csv"

# Contagens oficiais. Copias que circulam fora do Kaggle vem truncadas em
# 1.048.575 linhas — o limite do Excel — e perdem 43% dos dados.
LINHAS_ESPERADAS = {
    "fraudTrain.csv": 1_296_675,
    "fraudTest.csv": 555_719,
}

bq = bigquery.Client(project=PROJECT_ID)
gcs = storage.Client(project=PROJECT_ID)

print(f"Projeto : {PROJECT_ID}")
print(f"Bucket  : gs://{BUCKET_NAME}")
print(f"Regiao  : {REGION}")

## Bucket

Idempotente: se já existir, apenas confirma.

In [ ]:
bucket = gcs.lookup_bucket(BUCKET_NAME)
if bucket is None:
    bucket = gcs.create_bucket(BUCKET_NAME, location=REGION)
    print(f"Bucket criado: gs://{BUCKET_NAME} em {REGION}")
else:
    print(f"Bucket ja existe: gs://{BUCKET_NAME} em {bucket.location}")

## Datasets do Medallion

Três datasets, um por camada. `exists_ok=True` deixa a célula idempotente.

In [ ]:
for nome in DATASETS:
    ds = bigquery.Dataset(f"{PROJECT_ID}.{nome}")
    ds.location = BQ_LOCATION
    ds.description = f"Camada {nome} — FraudFlow Trabalho 1"
    bq.create_dataset(ds, exists_ok=True)
    print(f"  {nome:<8} ok")

print("\nDatasets prontos")

## Os arquivos já estão no bucket?

O download do Kaggle só é necessário na primeira execução.

In [ ]:
existentes = {b.name: b.size for b in gcs.list_blobs(BUCKET_NAME)
              if b.name.endswith('.csv')}

if existentes:
    for nome, tamanho in sorted(existentes.items()):
        print(f"  {nome:<34} {tamanho/1024**2:8.1f} MB")
else:
    print("bucket vazio — rode as celulas de download abaixo")

JA_INGERIDO = ARQUIVO_TRAIN in existentes and ARQUIVO_HOLDOUT in existentes
print(f"\nJa ingerido: {JA_INGERIDO}")

## Credencial do Kaggle

O Kaggle não baixa mais um `kaggle.json`. Hoje ele mostra um token começando com
`KGAT_` em **kaggle.com → Settings → API → Generate New Token**.

A célula abaixo pede o token com `getpass`: ele **não fica salvo no notebook**,
então não vaza para o Git quando você comitar. Se o kernel reiniciar, rode de novo.

Se a conta pedir verificação de telefone para liberar a API, faça em
kaggle.com/settings antes.

In [ ]:
if not JA_INGERIDO:
    os.environ["KAGGLE_API_TOKEN"] = getpass("Token do Kaggle (KGAT_...): ").strip()
    print("token registrado nesta sessao")
else:
    print("ja ingerido — pule esta celula")

## Download e descompactação

In [ ]:
TMP = Path('/content/kaggle') if Path('/content').exists() else Path('./kaggle_tmp')
TMP.mkdir(parents=True, exist_ok=True)

if not JA_INGERIDO:
    !kaggle datasets download -d kartik2112/fraud-detection -p {TMP} --force

    zips = list(TMP.glob('*.zip'))
    assert zips, 'o download nao produziu nenhum zip — confira o token'
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(TMP / 'extraido')

    arquivos = {p.name: p for p in (TMP / 'extraido').rglob('*.csv')}
    print("\nArquivos:", ", ".join(sorted(arquivos)))
else:
    arquivos = {}
    print("ja ingerido — pule esta celula")

## Conferência das contagens

Antes de subir qualquer coisa. Se algum arquivo vier truncado, o problema aparece
aqui e não três notebooks adiante.

In [ ]:
if not JA_INGERIDO:
    problemas = []
    for nome, esperado in LINHAS_ESPERADAS.items():
        obtido = sum(1 for _ in open(arquivos[nome], encoding='utf-8')) - 1
        ok = obtido == esperado
        print(f"  {nome:<16} {obtido:>9,} linhas  (esperado {esperado:>9,})  "
              f"{'OK' if ok else 'DIVERGENTE'}")
        if not ok:
            problemas.append(nome)

    assert not problemas, (
        f"contagem divergente em {problemas}. Provavel causa: copia truncada. "
        "As versoes fora do Kaggle param em 1.048.575 linhas, o limite do Excel."
    )
    print("\nContagens conferem")
else:
    print("ja ingerido — pule esta celula")

## Upload, com o holdout separado

Aqui a regra é aplicada: treino e holdout vão para prefixos diferentes.

In [ ]:
DESTINOS = {
    "fraudTrain.csv": ARQUIVO_TRAIN,
    "fraudTest.csv": ARQUIVO_HOLDOUT,
}

if not JA_INGERIDO:
    for nome, destino in DESTINOS.items():
        bucket.blob(destino).upload_from_filename(str(arquivos[nome]))
        marca = "  <- LACRADO, nao entra no BigQuery" if "holdout" in destino else ""
        print(f"  gs://{BUCKET_NAME}/{destino}{marca}")
else:
    print("ja ingerido — pule esta celula")

## Conferência final

O tamanho confirma que treino e holdout não trocaram de lugar: o treino é o
arquivo maior.

In [ ]:
objetos = {b.name: b.size for b in gcs.list_blobs(BUCKET_NAME) if b.name.endswith('.csv')}
for nome, tamanho in sorted(objetos.items()):
    print(f"  {nome:<34} {tamanho/1024**2:8.1f} MB")

assert ARQUIVO_TRAIN in objetos, f"faltando {ARQUIVO_TRAIN}"
assert ARQUIVO_HOLDOUT in objetos, f"faltando {ARQUIVO_HOLDOUT}"
assert objetos[ARQUIVO_TRAIN] > objetos[ARQUIVO_HOLDOUT], \
    "o treino deveria ser o MAIOR — train e holdout trocaram de lugar"

print("\nIngestao concluida.")
print(f"  Bronze le apenas : gs://{BUCKET_NAME}/{ARQUIVO_TRAIN}")
print(f"  Holdout intocado : gs://{BUCKET_NAME}/{ARQUIVO_HOLDOUT}")

---

**Próximo:** `01_raw_to_bronze.ipynb`

---

## Recomeçar do zero

A célula abaixo apaga as três camadas do BigQuery. O bucket e os CSVs
permanecem, então reconstruir custa apenas o tempo de rodar os notebooks 01 a 04.

Serve para verificar que o pipeline roda do zero, sem nenhum passo manual
anterior.

In [ ]:
APAGAR = False   # mude para True conscientemente

if APAGAR:
    for nome in DATASETS:
        bq.delete_dataset(f"{PROJECT_ID}.{nome}", delete_contents=True, not_found_ok=True)
        print(f"  {nome} apagado")
    print("\nRode os notebooks 00 a 04 de novo.")
else:
    print("nada foi apagado (APAGAR = False)")